# 02 · Feature EngineeringBuild the modelling matrix and sanity-check it for leakage.

In [ ]:
import sys, warningssys.path.insert(0, "..")warnings.filterwarnings("ignore")from src.config import Configfrom src.data.loader import load_datasetfrom src.data.validation import validatefrom src.data.cleaning import cleanconfig = Config.load(config_path="../config/config.yaml", model_params_path="../config/model_params.yaml")config.root = config.root  # notebooks run from notebooks/raw = load_dataset("../data/sample/sample_store_sales.csv", config, preset="favorita")report = validate(raw, config)frame, stats = clean(raw, config)print(report.passed, frame.shape, stats)

## Build features

In [ ]:
from src.features.builder import FeatureBuilderbuilder = FeatureBuilder(config)features = builder.drop_warmup(builder.fit_transform(frame))print(len(builder.spec.feature_names), "features;", features.shape, "rows after warm-up")builder.spec.feature_names

## Leakage checkRolling stats are computed on the shifted series: a spike must not appear in its own row.

In [ ]:
import pandas as pdfrom src.features.lags import add_rolling_featuresprobe = frame.head(60).copy().sort_values("date").reset_index(drop=True)probe["store"], probe["product"], probe["sales"] = "S", "P", 10.0probe.loc[30, "sales"] = 1000.0out = add_rolling_features(probe, [7], ["mean"])out.loc[29:32, ["date", "sales", "sales_roll7_mean"]]

## Correlation with the target

In [ ]:
numeric = features[builder.spec.feature_names + ["sales"]].select_dtypes("number")numeric.corr()["sales"].drop("sales").sort_values(key=abs, ascending=False).head(20)

## Walk-forward folds

In [ ]:
from src.validation.splitters import make_splits[f.describe() for f in make_splits(features, config)]